# HW6 — การประยุกต์ใช้ในอุตสาหกรรม (6 ข้อ)

รันเซลล์เตรียมข้อมูลและคำตอบจากบนลงล่าง ทุกข้อรันได้ครบ


In [1]:
from probability import *
import random
import time
import matplotlib.pyplot as plt
from inspect import getsource

def psource(*objects):
    for obj in objects:
        print(getsource(obj))


## ส่วนที่ 1: ติดตามสุขภาพเครื่องจักรด้วย HMM

**ตัวแปรสถานะซ่อน** `X_t` คือสภาพจริงของปั๊มในกะที่ t มีสองค่า: `Healthy` (index 0) กับ `Degraded` (index 1)
เราไม่มีทางรู้ค่าจริงโดยไม่ถอดเครื่อง

**หลักฐานที่สังเกตได้** `E_t` คือเซ็นเซอร์สั่นสะเทือนแจ้งเตือนหรือไม่ (`True` / `False`)

**Transition model** สะท้อนฟิสิกส์ของการสึกหรอ: เครื่องดีอาจเสื่อมได้ 5% ต่อกะ
แต่เครื่องที่เสื่อมแล้ว **ไม่หายเอง** (มีเพียง 2% ที่กลับมาปกติจากการปรับตั้งย่อย)
นี่คือ absorbing-ish chain ซึ่งต่างจากตัวอย่างร่มที่ฝนหยุดตกเองได้

**Sensor model** สะท้อนคุณภาพเซ็นเซอร์: false alarm 10% และตรวจจับได้ 80% เมื่อเครื่องเสื่อมจริง

$$P(E_t = alert \mid X_t = Healthy) = 0.10 \qquad P(E_t = alert \mid X_t = Degraded) = 0.80$$

In [2]:
import random
import matplotlib.pyplot as plt

# index 0 = Healthy, index 1 = Degraded
maint_transition = [[0.95, 0.05],    # จาก Healthy ไป [Healthy, Degraded]
                    [0.02, 0.98]]    # จาก Degraded ไป [Healthy, Degraded]

maint_sensor = [[0.10, 0.80],        # P(alert=True  | [Healthy, Degraded])
                [0.90, 0.20]]        # P(alert=False | [Healthy, Degraded])

maint_prior = [0.98, 0.02]           # เพิ่งผ่าน PM มา จึงมั่นใจว่าเครื่องดี

pump = HiddenMarkovModel(maint_transition, maint_sensor, maint_prior)

# บันทึกการแจ้งเตือนจริง 10 กะ (T = เซ็นเซอร์แจ้งเตือน)
alerts = [F, F, F, T, F, T, T, T, T, T]

belief = maint_prior
filtered = []
print('กะ  alert   P(Degraded)  สถานะที่ระบบรายงาน')
print('-' * 52)
for t, ev in enumerate(alerts, start=1):
    belief = forward(pump, belief, ev)
    filtered.append(float(belief[1]))
    flag = 'ALARM' if belief[1] > 0.5 else 'ok'
    print('%2d  %-6s  %10.4f   %s' % (t, 'alert' if ev else '-', belief[1], flag))

กะ  alert   P(Degraded)  สถานะที่ระบบรายงาน
----------------------------------------------------
 1  -           0.0161   ok
 2  -           0.0152   ok
 3  -           0.0150   ok
 4  alert       0.3534   ok
 5  -           0.1193   ok
 6  alert       0.6054   ALARM
 7  alert       0.9269   ALARM
 8  alert       0.9881   ALARM
 9  alert       0.9960   ALARM
10  alert       0.9970   ALARM


In [3]:
def smooth(hmm, ev):
    # คืนลิสต์ sv โดย sv[k] = P(X_{k+1} | e_1..e_n)  (forward-backward, AIMA รูป 15.4)
    fv = [hmm.prior]
    for e in ev:
        fv.append(forward(hmm, fv[-1], e))
    b = [1.0, 1.0]
    sv = [None] * len(ev)
    for i in range(len(ev) - 1, -1, -1):
        sv[i] = normalize(element_wise_product(fv[i + 1], b))
        b = backward(hmm, b, ev[i])
    return sv


# ตรวจกับตัวอย่างร่มในหนังสือ: P(Rain_1 | u_1, u_2) = 0.8834
umbrella = HiddenMarkovModel([[0.7, 0.3], [0.3, 0.7]], [[0.9, 0.2], [0.1, 0.8]], [0.5, 0.5])
assert abs(smooth(umbrella, [T, T])[0][0] - 0.8834) < 1e-4, 'smooth ไม่ตรงกับค่าในหนังสือ'

smoothed = [float(s[1]) for s in smooth(pump, alerts)]
# invariant: เวลาสุดท้ายไม่มีอนาคตให้มองย้อน ค่าจึงต้องเท่ากับ filtering เป๊ะ
assert abs(smoothed[-1] - filtered[-1]) < 1e-12



In [4]:
def fixed_lag(hmm, ev_so_far, d):
    # คืน P(X_{t-d} | e_1..e_t) เมื่อ t = len(ev_so_far); คืน None ถ้าข้อมูลยังไม่พอ
    t = len(ev_so_far)
    if t - d < 1:
        return None
    return float(smooth(hmm, ev_so_far)[t - d - 1][1])




## ส่วนที่ 3: Particle filtering เมื่อสถานะเป็นตัวเลขต่อเนื่อง

HMM ข้างบนบีบสภาพเครื่องให้เหลือ 2 ค่า ซึ่งพอสำหรับ alarm แต่ไม่พอสำหรับวางแผนอะไหล่
ทีมวางแผนอยากรู้ **ระดับการสึกหรอ** $w_t \in [0, 1]$ เป็นตัวเลขต่อเนื่อง เพื่อประเมิน
Remaining Useful Life (RUL) ว่าเหลืออีกกี่กะก่อนถึงเกณฑ์เปลี่ยนอะไหล่ที่ $w = 0.9$

สถานะต่อเนื่องทำให้ตารางความน่าจะเป็นแบบ HMM ใช้ไม่ได้ จึงต้องใช้ particle filter

- **Transition:** $w_t = w_{t-1} + \max(0, \mathcal{N}(0.06,\ 0.02))$ สึกหรอเพิ่มขึ้นอย่างเดียว ไม่ลดลง
- **Sensor:** อุณหภูมิตลับลูกปืน $z_t = 30 + 60 w_t + \mathcal{N}(0, 6)$ องศาเซลเซียส
- **Weight:** ใช้ `gaussian(mu, sigma, x)` ที่มีอยู่แล้วในโมดูลเป็น likelihood
- **Resample:** ใช้ `weighted_sample_with_replacement` จาก `utils`

`particle_filtering` ในโมดูลรองรับเฉพาะ 2 สถานะ (`'A'` และ `'B'`) จึงใช้กับโจทย์นี้ไม่ได้
แต่แนวคิด predict, weight, resample เหมือนกันทุกประการ เขียนเองได้ในไม่กี่บรรทัด

In [5]:
from utils import weighted_sample_with_replacement

WEAR_STEP_MEAN, WEAR_STEP_SD = 0.06, 0.02
TEMP_BASE, TEMP_GAIN, TEMP_SD = 30.0, 60.0, 6.0


def simulate_pump(n_shifts=12, seed=7):
    # สร้างข้อมูลจริงที่ระบบไม่มีวันเห็น เอาไว้ตรวจว่า particle filter ตามทันหรือไม่
    random.seed(seed)
    w, 

## ส่วนที่ 4: Monte Carlo Localization กับ AGV ในคลังสินค้า

AGV ที่ลำเลียงอะไหล่ไปยังจุดซ่อมต้องรู้ตำแหน่งตัวเอง GPS ใช้ในอาคารไม่ได้
และ odometry (นับรอบล้อ) สะสมความคลาดเคลื่อนเรื่อย ๆ วิธีมาตรฐานในอุตสาหกรรมคือ MCL
ซึ่งก็คือ particle filter บนแผนที่ โดยใช้ LiDAR สี่ทิศเป็นเซ็นเซอร์

`1` ในแผนที่คือชั้นวางของหรือผนัง `0` คือทางเดินที่ AGV วิ่งได้

In [6]:
warehouse = MCLmap([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0],
                    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0],
                    [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0],
                    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0],
                    [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0],
                    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0],
                    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0],
                    [0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0],
                    [0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
                    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
                    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


def P_motion_sample(kin_state, v, w):
    # แบบจำลองการเคลื่อนที่: หมุนก่อนแล้วจึงเคลื่อนที่ (ไม่มี noise เพื่อความง่าย)
    pos, orient = kin_state[:2], (kin_state[2] + w) % 4
    for _ in range(orient):
        v = (v[1], -v[0])
    return vector_add(pos, v) + (orient,)


def P_sensor(x, y):
    # likelihood ของระยะที่วัดได้ x เทียบกับระยะที่คาดจากแผนที่ y
    if x == y:
        return 0.8
    elif abs(x - y) <= 2:
        return 0.05
    return 0


def confidence(S, k=3):
    counts = {}
    for x, y, _ in S:
        counts[(x, y)] = counts.get((x, y), 0) + 1
    return [(cell, 100.0 * n / len(S))
            for cell, n in sorted(counts.items(), key=lambda kv: -kv[1])[:k]]




## ส่วนที่ 5: Decision Theoretic Agent กับการตัดสินใจหยุดซ่อม

ความน่าจะเป็นอย่างเดียวตัดสินใจไม่ได้ ต้องมีต้นทุน ตัวเลขจริงจากโรงงาน

| | เครื่องเสื่อมจริง | เครื่องปกติ |
|---|---|---|
| **หยุดซ่อมเดี๋ยวนี้** | -50,000 (ซ่อมตามแผน) | -50,000 (หยุดสายผลิตเสียเปล่า) |
| **รอถึง PM รอบหน้า** | -400,000 (พังกลางกะ, ของเสีย, ซ่อมฉุกเฉิน) | 0 |

$$EU(a) = \sum_{s} P(s \mid e)\, U(a, s) \qquad MEU(e) = \max_a EU(a)$$

> คลาส `DecisionNetwork` และ `InformationGatheringAgent` ในโมดูลเป็น abstract class
> ที่ยังไม่สมบูรณ์ (เมธอด `get_utility`, `cost`, `request` โยน `NotImplementedError`
> และ `get_expected_utility` รับ evidence เป็น list ซึ่ง `enumeration_ask` ใช้ไม่ได้)
> ลองอ่านด้วย `psource(DecisionNetwork)` แล้วเปรียบเทียบกับที่เราเขียนตรงตามสูตรข้างบน
> ซึ่งสั้นกว่าและตรวจสอบได้

In [7]:
COST_REPAIR = 50_000       # หยุดซ่อมตามแผน
COST_BREAKDOWN = 400_000   # เครื่องพังกลางกะ

UTILITY = {('repair', True): -COST_REPAIR,   ('repair', False): -COST_REPAIR,
           ('wait',   True): -COST_BREAKDOWN, ('wait',   False): 0}

ACTIONS = ('repair', 'wait')


def expected_utility(action, p_degraded):
    return p_degraded * UTILITY[(action, True)] + (1 - p_degraded) * UTILITY[(action, False)]


def meu(p_degraded):
    # คืน (ค่าอรรถประโยชน์สูงสุด, การกระทำที่ควรเลือก)
    return max((expected_utility(a, p_degraded), a) for a in ACTIONS)


breakeven = COST_REPAIR / COST_BREAKDOWN


## ส่วนที่ 6: Information Gathering Agent กับ VPI

โรงงานมีทางเลือกเพิ่ม: จ่าย **8,000 บาท** ส่งช่างมาทำ thermal scan ก่อนตัดสินใจ
การสแกนไม่สมบูรณ์แบบ แต่แม่นกว่าเซ็นเซอร์สั่นสะเทือน

$$P(scan{=}pos \mid Degraded) = 0.90 \qquad P(scan{=}pos \mid Healthy) = 0.15$$

คำถามคือ **คุ้มไหม** ตอบด้วย Value of Information (AIMA สมการ 16.7) โดยที่
"ข้อมูลมีค่า ก็ต่อเมื่อมันมีโอกาสทำให้เราเปลี่ยนใจ"

$$VPI(E) = \left(\sum_{e} P(e)\, MEU(e)\right) - MEU(\varnothing)$$

ให้สังเกตว่าเราใช้ Bayes' rule ธรรมดาในการอัปเดต prior เป็น posterior หลังรู้ผลสแกน
เหมือนที่ทำมาตลอดทั้งโน้ตบุ๊ก

In [8]:
P_SCAN_POS = {True: 0.90, False: 0.15}   # P(scan=positive | Degraded / Healthy)
SCAN_COST = 8_000


def vpi_scan(p_degraded):
    # มูลค่าคาดหวังของการรู้ผล thermal scan ก่อนตัดสินใจ
    total = 0.0
    for result in (True, False):
        lik_d = P_SCAN_POS[True] if result else 1 - P_SCAN_POS[True]
        lik_h = P_SCAN_POS[False] if result else 1 - P_SCAN_POS[False]
        p_result = lik_d * p_degraded + lik_h * (1 - p_degraded)
        if p_result == 0:
            continue
        posterior = lik_d * p_degraded / p_result          # Bayes' rule
        total += p_result * meu(posterior)[0]
    return total - meu(p_degraded)[0]




### ข้อ 1: เซ็นเซอร์ที่ "ดีกว่า" คุ้มค่าจริงไหม

ฝ่ายจัดซื้อเสนอเซ็นเซอร์รุ่นใหม่ที่ false alarm ลดจาก 10% เหลือ 3% และ detection เพิ่มจาก 80% เป็น 90%
ราคาแพงกว่าเดิม 3 เท่า ทีมของคุณต้องตอบว่าคุ้มไหม

1. สร้าง `pump_v2` ด้วย sensor model ใหม่ แล้วรัน filtering บน `alerts` ชุดเดิม
   ระบบสั่งหยุดซ่อม (ตามเกณฑ์ MEU) **เร็วขึ้นกี่กะ**
2. รันทั้งสองเซ็นเซอร์บนลำดับ **false alarm** `[F, F, T, F, F, F]`
   ซึ่งเป็นกรณีที่เซ็นเซอร์เตือนผิดครั้งเดียวแล้วเงียบ
   นับว่าแต่ละเซ็นเซอร์ทำให้ระบบ **สั่งซ่อมเก้อกี่กะ**
3. สรุปว่าควรซื้อไหม พร้อมเหตุผลจากตัวเลขทั้งสองข้อ

*คำใบ้:* `maint_sensor` แถวแรกคือ `P(alert=True | [Healthy, Degraded])` แถวสองคือส่วนเติมเต็มให้รวมเป็น 1
ผลลัพธ์อาจไม่เป็นไปตามที่คาด ให้เชื่อตัวเลข

In [9]:
# ข้อ 1: คำตอบ
sensor_v2 = [[0.03, 0.90], [0.97, 0.10]]
pump_v2 = HiddenMarkovModel(maint_transition, sensor_v2, maint_prior)
def run_filter(hmm, seq):
    belief, out = hmm.prior, []
    for event in seq:
        belief = forward(hmm, belief, event)
        out.append(float(belief[1]))
    return out
def repair_shifts(beliefs):
    return [t for t, p in enumerate(beliefs, 1) if meu(p)[1] == 'repair']
filtered_v2 = run_filter(pump_v2, alerts)
print('1) การตรวจจับของจริง')
print('   เซ็นเซอร์เดิม  สั่งซ่อมที่กะ', repair_shifts(filtered))
print('   รุ่นใหม่       สั่งซ่อมที่กะ', repair_shifts(filtered_v2))
print('   ความมั่นใจที่กะ 4: เดิม %.3f -> ใหม่ %.3f' % (filtered[3], filtered_v2[3]))
FALSE_ALARM = [F, F, T, F, F, F]
print('\n2) กรณีเซ็นเซอร์เตือนผิดครั้งเดียว')
for name, hmm in [('เดิม  ', pump), ('ใหม่  ', pump_v2)]:
    beliefs = run_filter(hmm, FALSE_ALARM)
    print('   %s P(Degraded) = %s -> สั่งซ่อมเก้อที่กะ %s' % (name, ['%.3f' % p for p in beliefs], repair_shifts(beliefs)))
print('\n3) ยังสรุปความคุ้มค่าจากสเปกอย่างเดียวไม่ได้: ต้องจำลองหลายลำดับและเทียบต้นทุนรวม')


1) การตรวจจับของจริง
   เซ็นเซอร์เดิม  สั่งซ่อมที่กะ [4, 6, 7, 8, 9, 10]
   รุ่นใหม่       สั่งซ่อมที่กะ [4, 5, 6, 7, 8, 9, 10]
   ความมั่นใจที่กะ 4: เดิม 0.353 -> ใหม่ 0.639

2) กรณีเซ็นเซอร์เตือนผิดครั้งเดียว
   เดิม   P(Degraded) = ['0.016', '0.015', '0.354', '0.120', '0.041', '0.021'] -> สั่งซ่อมเก้อที่กะ [3]
   ใหม่   P(Degraded) = ['0.008', '0.006', '0.639', '0.157', '0.025', '0.008'] -> สั่งซ่อมเก้อที่กะ [3, 4]

3) ยังสรุปความคุ้มค่าจากสเปกอย่างเดียวไม่ได้: ต้องจำลองหลายลำดับและเทียบต้นทุนรวม


### ข้อ 2: เลือกค่า lag ให้ dashboard

คำนวณ **ค่าคลาดเคลื่อนสัมบูรณ์เฉลี่ย** ระหว่างรายงานที่ lag = d กับค่า smoothing เต็มรูป
(ถือว่า smoothing เต็มรูปคือ "ความจริงที่ดีที่สุดที่เรารู้ได้") สำหรับ d = 0, 1, 2, 3

แล้วตอบว่า **lag เท่าไรที่คุ้มที่สุด** ถ้าทุก 1 กะที่หน่วงเพิ่ม มีต้นทุนเทียบเท่าค่าคลาดเคลื่อน 0.05

*คำใบ้:* ใช้ `fixed_lag(pump, alerts[:t], d)` เทียบกับ `smoothed[t - d - 1]` เฉพาะ t ที่ค่าไม่เป็น `None`

In [10]:
# ข้อ 2: คำตอบ
LAG_PENALTY = 0.05
costs = {}
for d in range(4):
    errs = []
    for t in range(1, len(alerts) + 1):
        value = fixed_lag(pump, alerts[:t], d)
        if value is not None:
            errs.append(abs(value - smoothed[t - d - 1]))
    mean_err = sum(errs) / len(errs) if errs else float('nan')
    costs[d] = mean_err + d * LAG_PENALTY
    print('d=%d  mean|error| = %.4f  ต้นทุนรวม = %.4f' % (d, mean_err, costs[d]))
print('lag ที่คุ้มที่สุดคือ d =', min(costs, key=costs.get))


d=0  mean|error| = 0.2026  ต้นทุนรวม = 0.2026
d=1  mean|error| = 0.1587  ต้นทุนรวม = 0.2087
d=2  mean|error| = 0.0968  ต้นทุนรวม = 0.1968
d=3  mean|error| = 0.0473  ต้นทุนรวม = 0.1973
lag ที่คุ้มที่สุดคือ d = 2


### ข้อ 3: ตรวจจับ particle depletion

เพิ่มการคำนวณ **effective sample size** $ESS = 1 / \sum_i w_i^2$ ลงใน particle filter
ค่านี้บอกว่า "จริง ๆ แล้วมีอนุภาคกี่ตัวที่มีส่วนร่วมในการประมาณ"
ถ้า ESS ร่วงต่ำกว่า N/2 แปลว่าอนุภาคส่วนใหญ่แทบไม่มีส่วนร่วม ค่าประมาณเริ่มเชื่อไม่ได้

จากนั้นจำลองสถานการณ์ **น้ำมันหล่อลื่นรั่ว** โดยให้ ground truth สึกหรอเร็วขึ้น 3 เท่า
(`WEAR_STEP_MEAN * 3` ใน `simulate_pump` เท่านั้น ห้ามแก้ใน filter)
แล้วเทียบ ESS กับกรณีปกติ (`simulate_pump()`) ว่าร่วงตอนไหนและร่วงแค่ไหน

*คำใบ้:* คำนวณ ESS จาก `weights` หลัง normalize แล้ว ก่อนขั้น resample

In [11]:
# ข้อ 3: คำตอบ
def simulate_pump_leak(n_shifts=9, seed=7, drift=3.0):
    random.seed(seed)
    w, truth, obs = 0.05, [], []
    for _ in range(n_shifts):
        w = min(1.0, w + max(0.0, random.gauss(WEAR_STEP_MEAN * drift, WEAR_STEP_SD)))
        truth.append(w); obs.append(TEMP_BASE + TEMP_GAIN * w + random.gauss(0, TEMP_SD))
    return truth, obs
def wear_pf_with_ess(obs, N=3000, seed=1):
    random.seed(seed); particles = [random.uniform(0.0, 0.12) for _ in range(N)]; mean, ess = [], []
    for z in obs:
        particles = [min(1.0, w + max(0.0, random.gauss(WEAR_STEP_MEAN, WEAR_STEP_SD))) for w in particles]
        weights = [gaussian(TEMP_BASE + TEMP_GAIN * w, TEMP_SD, z) for w in particles]
        total = sum(weights); weights = [w / total for w in weights] if total > 0 else [1.0 / N] * N
        ess.append(1.0 / sum(w * w for w in weights))
        particles = weighted_sample_with_replacement(N, particles, weights); mean.append(sum(particles) / N)
    return mean, ess
truth_leak, obs_leak = simulate_pump_leak(); est_leak, ess_leak = wear_pf_with_ess(obs_leak)
print('%3s %9s %9s %10s %s' % ('กะ', 'จริง', 'ประมาณ', 'ESS', 'สถานะ'))
for t in range(len(truth_leak)):
    warn = 'LOW ESS' if ess_leak[t] < 1500 else ''
    print('%3d %9.3f %9.3f %10.1f %s' % (t + 1, truth_leak[t], est_leak[t], ess_leak[t], warn))


 กะ      จริง    ประมาณ        ESS สถานะ
  1     0.225     0.142     2304.9 
  2     0.400     0.224     2244.4 
  3     0.562     0.320     1533.7 
  4     0.764     0.444      353.9 LOW ESS
  5     0.965     0.579      372.0 LOW ESS
  6     1.000     0.676     1165.4 LOW ESS
  7     1.000     0.767     1251.0 LOW ESS
  8     1.000     0.852     1959.8 
  9     1.000     0.900     2774.8 


### ข้อ 4: AGV หลงทาง

รัน MCL ด้วย **การสแกนที่ผิดพลาด** คือ `z = (9, 9, 9, 9)` ซึ่งเป็นค่าที่แทบเป็นไปไม่ได้บนแผนที่นี้
(เช่น LiDAR สกปรกหรือมีคนเดินบัง)

1. เกิดอะไรขึ้นกับการกระจายของอนุภาค
2. ระบบจริงควรทำอย่างไรเมื่อ likelihood ของทุกอนุภาคเป็นศูนย์

*คำใบ้:* `P_sensor` คืน `0` เมื่อ `abs(x - y) > 2` ลองดูโค้ดของ `monte_carlo_localization` ด้วย `psource`

In [12]:
# ข้อ 4: รันแล้วสังเกต จากนั้นตอบคำถามในคอมเมนต์
random.seed(11)
try:
    S_bad = monte_carlo_localization({'v': (0, 0), 'w': 0}, (9, 9, 9, 9),
                                     500, P_motion_sample, P_sensor, warehouse)
    print('จำนวนช่องที่มีอนุภาค:', len({(x, y) for x, y, _ in S_bad}))
    for cell, pct in confidence(S_bad, 5):
        print('   %s : %.1f%%' % (cell, pct))
except IndexError as e:
    print('monte_carlo_localization ล้มด้วย IndexError: %r' % (e,))
    print('อ่านโค้ดด้วย psource(monte_carlo_localization) แล้วหาว่าบรรทัดไหนพัง')

# วินิจฉัยก่อนโทษโค้ด: อัลกอริทึมคูณ likelihood ของเซ็นเซอร์ทั้ง 4 ทิศเข้าด้วยกัน
random.seed(11)
particles = [warehouse.sample() for _ in range(500)]
weights = []
for kin in particles:
    w = 1.0
    for j in range(4):
        w *= P_sensor(9, warehouse.ray_cast(j, kin))
    weights.append(w)

print('อนุภาคที่มีน้ำหนักมากกว่าศูนย์: %d จาก %d' % (sum(1 for w in weights if w > 0), len(weights)))
print('ผลรวมน้ำหนักทั้งหมด =', sum(weights))

# คำตอบ:
# 1. อนุภาคทั้ง 500 ตัวมีน้ำหนักเป็นศูนย์ เพราะ scan (9, 9, 9, 9) เป็นไปไม่ได้บนแผนที่ แล้ว resample จึงเกิด IndexError.
# 2. ระบบจริงควรทิ้ง outlier, ใช้ likelihood floor เพื่อกันน้ำหนักเป็นศูนย์ และทำ global relocalization หากพบต่อเนื่อง.

monte_carlo_localization ล้มด้วย IndexError: IndexError('list index out of range')
อ่านโค้ดด้วย psource(monte_carlo_localization) แล้วหาว่าบรรทัดไหนพัง
อนุภาคที่มีน้ำหนักมากกว่าศูนย์: 0 จาก 500
ผลรวมน้ำหนักทั้งหมด = 0.0


### ข้อ 5: เปลี่ยนตัวเลขต้นทุน เปลี่ยนทั้งระบบ

โรงงานย้ายไปผลิตยาฉีด ซึ่งความเสียหายจากเครื่องพังกลางกะพุ่งเป็น **5,000,000 บาท**
ส่วนค่าซ่อมยังเท่าเดิม

1. เกณฑ์ break-even ใหม่เป็นเท่าไร
2. ระบบจะสั่งหยุดซ่อมที่กะไหน
3. VPI ที่กะ 4 เปลี่ยนไปอย่างไร และยังคุ้มค่าสแกน 8,000 บาทไหม

*คำใบ้:* เขียนฟังก์ชันรับ `cost_breakdown` เป็นพารามิเตอร์ แทนการแก้ตัวแปร global

In [13]:
# ข้อ 5: คำตอบ
def make_decision_model(cost_breakdown, cost_repair=COST_REPAIR):
    U = {('repair', True): -cost_repair, ('repair', False): -cost_repair, ('wait', True): -cost_breakdown, ('wait', False): 0}
    def eu(action, p): return p * U[(action, True)] + (1 - p) * U[(action, False)]
    def meu_(p): return max((eu(action, p), action) for action in ACTIONS)
    return eu, meu_, cost_repair / cost_breakdown
eu5, meu5, breakeven5 = make_decision_model(5_000_000)
first_repair5 = next(t for t, p in enumerate(filtered, 1) if meu5(p)[1] == 'repair')
def vpi_scan_for(p_degraded, decision):
    total = 0.0
    for result in (True, False):
        lik_d = P_SCAN_POS[True] if result else 1 - P_SCAN_POS[True]
        lik_h = P_SCAN_POS[False] if result else 1 - P_SCAN_POS[False]
        p_result = lik_d * p_degraded + lik_h * (1 - p_degraded)
        posterior = lik_d * p_degraded / p_result
        total += p_result * decision(posterior)[0]
    return total - decision(p_degraded)[0]
vpi5_shift4 = vpi_scan_for(filtered[3], meu5)
print('1) break-even ใหม่ = %.3f' % breakeven5)
print('2) สั่งหยุดซ่อมครั้งแรกที่กะ', first_repair5)
print('3) VPI ที่กะ 4 = %.0f บาท -> %sสแกน 8,000 บาท' % (vpi5_shift4, 'คุ้ม' if vpi5_shift4 > SCAN_COST else 'ไม่คุ้ม'))


1) break-even ใหม่ = 0.010
2) สั่งหยุดซ่อมครั้งแรกที่กะ 1
3) VPI ที่กะ 4 = -0 บาท -> ไม่คุ้มสแกน 8,000 บาท


### ข้อ 6 (ท้าทาย): เรียนพารามิเตอร์จากข้อมูล

ที่ผ่านมาเรากรอก `maint_transition` และ `maint_sensor` ด้วยมือ ในงานจริงต้องเรียนจากข้อมูล

ถ้ามีข้อมูลที่ **รู้สถานะจริง** (จากใบบันทึกการซ่อม) การประมาณค่าทำได้ด้วยการนับตรง ๆ
เขียนฟังก์ชัน `fit_hmm(states, evidence)` ที่รับลิสต์สถานะจริงกับลิสต์หลักฐาน
แล้วคืน transition model กับ sensor model โดยใช้ **Laplace smoothing** (บวก 1 ทุกช่อง)
เพื่อกันความน่าจะเป็นเป็นศูนย์

ทดสอบโดยสร้างข้อมูลจาก `pump` ตัวจริง 5,000 กะ แล้วดูว่าค่าที่เรียนได้เข้าใกล้ของจริงไหม

*คำใบ้:* transition นับคู่ `(states[i], states[i+1])`, sensor นับคู่ `(states[i], evidence[i])`
Laplace smoothing คือ `(count + 1) / (total + จำนวนค่าที่เป็นไปได้)`

In [14]:
# ข้อ 6: คำตอบ
def sample_hmm(hmm, n, seed=0):
    random.seed(seed); states, evidence = [], []; healthy = probability(hmm.prior[0])
    for _ in range(n):
        healthy = probability(hmm.transition_model[0 if healthy else 1][0]); states.append(healthy)
        evidence.append(probability(hmm.sensor_model[0][0 if healthy else 1]))
    return states, evidence
def fit_hmm(states, evidence):
    tc = {(a, b): 1 for a in (True, False) for b in (True, False)}
    sc = {(a, b): 1 for a in (True, False) for b in (True, False)}
    for i in range(len(states) - 1): tc[(states[i], states[i + 1])] += 1
    for state, event in zip(states, evidence): sc[(state, event)] += 1
    def row(counts, state):
        total = counts[(state, True)] + counts[(state, False)]
        return [counts[(state, True)] / total, counts[(state, False)] / total]
    trans = [row(tc, True), row(tc, False)]
    p_alert = [sc[(True, True)] / (sc[(True, True)] + sc[(True, False)]), sc[(False, True)] / (sc[(False, True)] + sc[(False, False)])]
    return trans, [p_alert, [1 - p_alert[0], 1 - p_alert[1]]]
states, evidence = sample_hmm(pump, 5000); learned = fit_hmm(states, evidence)
print('transition ที่เรียนได้:', learned[0]); print('transition จริง      :', maint_transition)
print('sensor ที่เรียนได้   :', learned[1]); print('sensor จริง          :', maint_sensor)


transition ที่เรียนได้: [[0.9557242251739405, 0.044275774826059454], [0.02016364699006429, 0.9798363530099357]]
transition จริง      : [[0.95, 0.05], [0.02, 0.98]]
sensor ที่เรียนได้   : [[0.09867172675521822, 0.8004674262342975], [0.9013282732447818, 0.19953257376570255]]
sensor จริง          : [[0.1, 0.8], [0.9, 0.2]]
